In [194]:
import pandas as pd
import numpy as np
import glob
import os
import re, math, unicodedata
from pathlib import Path

In [195]:
# Define file paths
PATH_SELLER_A = 'data/seller-a/'
PATH_SELLER_B = 'data/seller-b/nish_catalog_run.csv'
PATH_SELLER_C = 'data/seller-c/jngems_custom_ignore.csv'
PATH_SELLER_D = 'data/seller-d/ratnapura.csv'
PATH_MASTER_DATASET = 'data/master_gem_schema.csv'
PATH_MASTER_CLEAN_DATASET = 'data/master_gem_schema_clean.csv'

# Define currency conversion rate
USD_TO_LKR = 330.0

# Define regex pattern for identifying pairs or sets
PAIR_PAT = re.compile(r"\b(pair|set|lot|parcel|pcs|pieces|two\s*stones?|matching|couple)\b", re.I)

# Initialize list to hold dataframes
frames = []

# Seller A

In [196]:
all_files = glob.glob(os.path.join(PATH_SELLER_A, '*.csv'))

# Exclude the master file if it already exists (to avoid self-merging)
master_file = os.path.join(PATH_SELLER_A, 'master-a.csv')
all_files = [f for f in all_files if f != master_file]

# Merge all CSVs
dataframes = [pd.read_csv(file) for file in all_files]
merged_df = pd.concat(dataframes, ignore_index=True)

# If master file exists, delete it first (optional safety step)
if os.path.exists(master_file):
    os.remove(master_file)

merged_df.drop(columns=['url'], inplace=True)
merged_df.drop(columns=['raw_description_text'], inplace=True)



In [197]:
merged_df.info()
merged_df['shape_cut'].unique()

def clean_shape(shape):
    """
    Normalize gem shape/cut values.
    Removes noise words (Treatment, Certificate, SKU, Enhancement, Colour, etc.)
    and maps to a canonical set.
    """
    if pd.isna(shape): 
        return np.nan
    s = str(shape).lower().strip()

    # Remove unwanted keywords
    s = re.sub(r"\b(treatment|certificate|sku|enhancement|colour|mixed|cartificate)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Canonical mapping
    canon = {
        "round brilliant": "Round",
        "round flower cut": "Round",
        "round flower": "Round",
        "round": "Round",
        "oval step cut": "Oval",
        "oval step": "Oval",
        "oval mixed cut": "Oval",
        "oval mixed": "Oval",
        "oval step treatment": "Oval",
        "oval": "Oval",
        "cushion step cut": "Cushion",
        "cushion shape": "Cushion",
        "square cushion": "Cushion",
        "rectangular cushion": "Cushion",
        "cushion mixed": "Cushion",
        "cushion": "Cushion",
        "pear step cut": "Pear",
        "pear shape": "Pear",
        "pear": "Pear",
        "trilliant step": "Trillion",
        "trillion": "Trillion",
        "trigonal": "Trillion",
        "baguete": "Baguette",
        "baguette": "Baguette",
        "princess cut": "Princess",
        "princess": "Princess",
        "square princess": "Princess",
        "octagon emerald": "Emerald Cut",
        "octagon": "Emerald Cut",
        "emerald": "Emerald Cut",
        "radiant cut": "Radiant",
        "radiant": "Radiant",
        "heart": "Heart",
        "hexagon": "Hexagon",
        "step": "Step Cut",
        "briolette": "Briolette",
        "cabochon": "Cabochon",
        "fancy": "Fancy",
        "brilliant": "Brilliant",
        "mix": "Mixed",
    }

    for key, val in canon.items():
        if key in s:
            return val

    return s.title()

merged_df["shape_cut"] = merged_df["shape_cut"].apply(clean_shape)
print(merged_df["shape_cut"].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          335 non-null    object 
 1   price_text    335 non-null    object 
 2   price_value   335 non-null    float64
 3   currency      335 non-null    object 
 4   gem_type      335 non-null    object 
 5   weight_carat  321 non-null    object 
 6   clarity       278 non-null    object 
 7   size_mm       271 non-null    object 
 8   colour        325 non-null    object 
 9   shape_cut     335 non-null    object 
 10  treatment     200 non-null    object 
 11  scraped_at    335 non-null    object 
dtypes: float64(1), object(11)
memory usage: 31.5+ KB
shape_cut
Oval           131
Cushion         76
Pear            39
Round           35
Emerald Cut     13
Heart           10
Baguette         7
Trillion         6
Brilliant        3
Princess         2
Mixed            2
Hexagon          2
C

In [198]:
merged_df['treatment'].unique()
def clean_treatment(t):
    """
    Normalize treatment values.
    Removes noise (certificate, videos, etc.) and maps to canonical categories.
    """
    if pd.isna(t): 
        return np.nan
    s = str(t).lower().strip()

    # remove noise words
    s = re.sub(r"(certificate|certificacte|videos?.*|upon request.*)", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    # canonical mapping
    if s in ("unheated","no heat","none"):
        return "Unheated"
    if s in ("heated","heat"):
        return "Heated"

    # fallback → capitalize cleaned string
    return s.title() if s else np.nan

merged_df["treatment"] = merged_df["treatment"].apply(clean_treatment)
print(merged_df["treatment"].value_counts())


treatment
Heated      127
Unheated     73
Name: count, dtype: int64


In [199]:
merged_df['clarity'].unique()
def clean_clarity(c):
    """
    Map messy clarity strings to short codes:
      - EC = Eye Clean / Clean
      - LC = Loupe Clean / Loop Clean (typo)
    Ignores junk words like 'Size', 'Colour', weird spacing, etc.
    Returns np.nan if nothing recognizable.
    """
    if c is None or (isinstance(c, float) and np.isnan(c)):
        return np.nan

    s = str(c).strip().lower()

    # fix common spacing glitches like "s ize"
    s = re.sub(r"\s+", " ", s)
    s = s.replace("s ize", "size")

    # drop non-signal words
    s = re.sub(r"\b(size|colour|color)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    # loupe/loop clean -> LC
    if re.search(r"\b(loupe|loop)\s*clean\b", s):
        return "LC"

    # eye clean or just 'clean' -> EC
    if re.search(r"\beye\s*clean\b", s) or re.fullmatch(r"clean", s):
        return "EC"

    # also accept 'eye clean <anything>' and '<anything> clean' as EC
    if "eye clean" in s or re.search(r"\bclean\b", s):
        return "EC"

    return np.nan

merged_df["clarity"] = merged_df["clarity"].apply(clean_clarity)
print(merged_df["clarity"].value_counts())

clarity
EC    229
LC     49
Name: count, dtype: int64


In [200]:
merged_df['colour'].unique()

def clean_colour(c):
    """
    Normalize gem colours.
    - Removes words like 'Shape', 'Size', 'Origin'
    - Fixes common typos (Cornlflower, Conflour -> Cornflower)
    - Title-cases colour names
    """
    if pd.isna(c): 
        return np.nan
    s = str(c).strip()

    # remove noise words
    s = re.sub(r"\b(shape|size|origin|colour|color)\b", "", s, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip()

    # fix typos for cornflower
    s = re.sub(r"cornl?f?l?ower", "Cornflower", s, flags=re.I)

    # normalize casing
    s = s.title()

    return s if s else np.nan

merged_df["colour"] = merged_df["colour"].apply(clean_colour)
print(merged_df["colour"].value_counts())

def fix_good_colour(row):
    c = str(row["colour"]).lower()
    if "good" in c:
        # try to infer colour from name
        name = str(row["name"]).lower()
        if "blue" in name:    return "Blue"
        if "pink" in name:    return "Pink"
        if "yellow" in name:  return "Yellow"
        if "green" in name:   return "Green"
        if "purple" in name:  return "Purple"
        if "orange" in name:  return "Orange"
        if "peach" in name:   return "Peach"
        if "padparadscha" in name: return "Padparadscha"
        # fallback
        return np.nan
    return row["colour"]

merged_df["colour"] = merged_df.apply(fix_good_colour, axis=1)
print(merged_df["colour"].value_counts())


colour
Royal Blue               56
Cornflower Blue          52
Vivid Pink               27
Yellow                   23
Medium Blue              17
Pinkish Orange           16
Red                      14
Green                    12
Cornflower               12
Medium Pink               9
Good                      9
Blue                      7
Orange Pinkish            6
Pink                      6
Hot Pink                  6
Corn Flower               5
Pinkish Red               4
Vivid Red                 4
Vivid Royal Blue          3
Orange                    3
Greenish Blue             3
Red Ruby                  2
Orangish Pink             2
Pink Spphire              2
Dark Blue                 2
Medium Yellow             2
Orangish Yellow           2
Green Blue                2
Light Blue                2
Bluish Green              1
Purplish Pink             1
Medium                    1
Pinkish Purple            1
Peach                     1
Conflour Blue             1
Sunny Yellow 

In [201]:
merged_df.rename(columns={
    "scraped_at": "collected_at"
}, inplace=True)   

In [202]:
def infer_treatment_from_name(row):
    # If treatment is already present, keep it
    if pd.notna(row["treatment"]) and str(row["treatment"]).strip():
        return row["treatment"]
    name = str(row["name"]).lower()
    # Look for common treatment keywords
    if "unheated" in name or "no heat" in name:
        return "Unheated"
    if "heated" in name or "heat" in name:
        return "Heated"
    return np.nan

merged_df["treatment"] = merged_df.apply(infer_treatment_from_name, axis=1)
print(merged_df["treatment"].value_counts(dropna=False))

treatment
Heated      127
Unheated    105
NaN         103
Name: count, dtype: int64


### Save Selle A Merge data


In [203]:
for col in merged_df.columns:
    unique_count = merged_df[col].nunique(dropna=False)
    print(f"{col}: {unique_count} unique values")
    print(merged_df[col].value_counts(dropna=False))


name: 306 unique values
name
1.02ct Natural Blue Sapphire            3
1.50ct Natural Blue Sapphire            3
2.28ct Natural Blue Sapphire            2
1.47ct Natural Unheated Padparadscha    2
1.09ct Natural Blue Sapphire            2
                                       ..
2.01ct Natural Blue Sapphire            1
1.99ct Natural Blue Sapphire            1
2.16ct Natural Blue Sapphire            1
2.25ct Natural Blue Sapphire            1
1.32ct Natural Unheated Ruby            1
Name: count, Length: 306, dtype: int64
price_text: 227 unique values
price_text
Rs 2,227,800.00    7
Rs 467,900.00      5
Rs 468,300.00      4
Rs 189,600.00      4
Rs 735,800.00      4
                  ..
Rs 891,200.00      1
Rs 1,470,500.00    1
Rs 1,559,600.00    1
Rs 1,827,000.00    1
Rs 441,500.00      1
Name: count, Length: 227, dtype: int64
price_value: 227 unique values
price_value
2227800.0    7
467900.0     5
468300.0     4
189600.0     4
735800.0     4
            ..
891200.0     1
1470500.0  

In [204]:

# Save new master file (always overwrite)
merged_df.to_csv(master_file, index=False)
print("Merged Seller A DataFrame. Shape:", merged_df.shape)

# Reload for check
df_a = pd.read_csv(master_file)
print(df_a.head())

PATH_SELLER_A = 'data/seller-a/master-a.csv'

Merged Seller A DataFrame. Shape: (335, 12)
                                         name     price_text  price_value  \
0       1.16ct Natural Unheated Pink Sapphire  Rs 713,500.00     713500.0   
1       1.30ct Natural Unheated Pink Sapphire  Rs 590,900.00     590900.0   
2       0.77ct Natural Unheated Pink Sapphire  Rs 356,800.00     356800.0   
3       0.89ct Natural Unheated Pink Sapphire  Rs 412,500.00     412500.0   
4  0.93ct Natural Unheated Pink Sapphire Pair  Rs 245,300.00     245300.0   

  currency                        gem_type weight_carat clarity size_mm  \
0      LKR  Natural Unheated Pink Sapphire         1.16      EC     NaN   
1      LKR  Natural Unheated Pink Sapphire         1.30      EC     NaN   
2      LKR           Natural Pink Sapphire         0.77      EC     NaN   
3      LKR           Natural Pink Sapphire         0.89      EC     NaN   
4      LKR  Natural Unheated Pink Sapphire         0.93     NaN    5 mm   

         colour shape_cut treatment       

In [205]:
def origin_filter(origin):
    if pd.isna(origin) or str(origin).strip() == "":
        return "Sri Lanka"   # default if blank
    s = str(origin).lower()
    if "sri lanka" in s or "ceylon" in s:
        return "Sri Lanka"
    return None   # mark for removal

# Seller B

In [206]:
seller_b_df = pd.read_csv(PATH_SELLER_B)
print("Original Seller B DataFrame. Shape:", seller_b_df.shape)


columns_to_drop = ['url', 'sold_out_flag', 'cut_grade', 'transparency', 'luster']
existing_columns_to_drop = [col for col in columns_to_drop if col in seller_b_df.columns]
if len(existing_columns_to_drop) > 0:
    print(f"Dropping columns from Seller B: {existing_columns_to_drop}")
    seller_b_df.drop(columns=existing_columns_to_drop, inplace=True)
else:
    print("No columns to drop from Seller B.")


seller_b_df['treatment'] = seller_b_df['treatment'].apply(lambda x: 'Unheated' if pd.isna(x) or str(x).strip().lower() == 'unheated' else x)
print(seller_b_df['treatment'].value_counts(dropna=False))


# Filter seller_b_df to keep only rows where origin is "Sri Lanka" or "Ceylon"
seller_b_df = seller_b_df[seller_b_df['origin'].str.lower().str.contains('sri lanka|ceylon', na=False)]
print("Filtered Seller B DataFrame. Shape:", seller_b_df.shape)
seller_b_df['origin'].value_counts()

seller_b_df.rename(columns={
    "scraped_at": "collected_at"
}, inplace=True)   


Original Seller B DataFrame. Shape: (13, 15)
No columns to drop from Seller B.
treatment
Unheated    7
Heated      6
Name: count, dtype: int64
Filtered Seller B DataFrame. Shape: (13, 15)


## Save Selle B data frame

In [207]:
seller_b_df.to_csv(PATH_SELLER_B, index=False)

print("Modified Seller B DataFrame. Shape:", seller_b_df.shape)
seller_b_df

Modified Seller B DataFrame. Shape: (13, 15)


,name,price_text,price_value,currency,colour,weight_cts,dimensions,category,shape,origin,treatment,clarity,hardness,certificate,collected_at
0,1.02 Ct Natural Unheated Yellow Sapphire Oval ...,$ 480.00,480.0,USD,Yellow,1.02,6.85 x 5.06 x 3.80,Sapphire,Oval,Sri Lanka,Unheated,VVS,9.0,Not yet,2025-09-01T15:16:19.608763+00:00
1,1.57 Ct Natural Blue Sapphire Oval Cut Loose G...,$ 450.00,450.0,USD,Blue,1.57,7.48 x 5.38 x 4.15,Sapphire,Oval,Sri Lanka,Heated,VS,9.0,Not yet,2025-09-01T15:18:17.938387+00:00
2,Natural Cornflower Blue Sapphire 1.21 Cts Oval...,$ 689.00,689.0,USD,Blue,1.21,7.14 x 5.93 x 3.62,Sapphire,Oval,Sri Lanka,Heated,VS,9.0,Not yet,2025-09-01T15:18:47.443768+00:00
3,5.20 Cts Bi-Color Natural Sapphire Cushion Cut...,"$ 3,950.00",3950.0,USD,Bi-Color,5.20,10.06 x 7.65 x 6.71,Sapphire,Cushion,Sri Lanka,Heated,VVS,9.0,Not yet,2025-09-01T15:19:45.088072+00:00
4,Natural Blue Sapphire 1.54 Cts Oval Shape Loos...,$ 590.00,590.0,USD,Blue,1.54,7.44 x 5.88 x 4.46,Sapphire,Oval,Sri Lanka,Heated,VVS,9.0,Not yet,2025-09-01T15:24:50.289359+00:00
5,2.24 Cts Cushion Blue Natural Sapphire Elegant...,$ 500.00,500.0,USD,Blue,2.24,6.75 x 6.65 x 5.09,Sapphire,Cushion,Sri Lanka,Heated,VS,9.0,Not yet,2025-09-01T15:28:46.165180+00:00
6,Unheated 0.47 Cts Cushion Bi-Color Natural Sap...,$ 150.00,150.0,USD,Bi-Color,0.47,4.79 x 4.02 x 2.58,Sapphire,Cushion,Sri Lanka,Unheated,VS,9.0,Not yet,2025-09-01T15:37:24.719690+00:00
7,Natural Unheated Purple Sapphire 1.58 Cts Oval...,$ 800.00,800.0,USD,Purple,1.58,6.64 x 5.91 x 4.99,Sapphire,Oval,Sri Lanka,Unheated,VVS,9.0,Not yet,2025-09-01T15:44:03.202330+00:00
8,Natural Unheated Yellow Sapphire 1.47 Cts Cush...,$ 360.00,360.0,USD,Yellow,1.47,7.87 x 5.17 x 3.96,Sapphire,Cushion,Sri Lanka,Unheated,VVS,9.0,Not yet,2025-09-01T15:44:27.189597+00:00
9,Natural Unheated Yellow Sapphire 1.36 Cts Emer...,$ 300.00,300.0,USD,Yellow,1.36,6.30 x 5.62 x 3.80,Sapphire,Emerald,Sri Lanka,Unheated,VS,9.0,Not yet,2025-09-01T15:45:34.588782+00:00
